# Парсинг PDF статей с Хабра → CSV

**Что делает скрипт:**
1. Читает все PDF из папки `PDF/`
2. Извлекает метаданные (заголовок, автор, хабы, время чтения, просмотры, тип, компания, рейтинг)
3. Сохраняет полный текст каждой статьи
4. Записывает всё в `dataset.csv`

**Требования:** `pip install PyMuPDF pandas`

In [ ]:
# Установка зависимостей (выполнить один раз)
# !pip install PyMuPDF pandas

In [ ]:
import fitz  # PyMuPDF
import pandas as pd
import glob
import re
import os


In [ ]:
# НАСТРОЙКИ
PDF_FOLDER = r"pdf_articles"      # папка с PDF-файлами
OUTPUT_CSV = r"dataset.csv"      # куда сохранить результат

all_pdf = sorted(glob.glob(os.path.join(PDF_FOLDER, "*.pdf")))
print(f"Найдено PDF: {len(all_pdf)}")
for p in all_pdf[:10]:
    print(" ", p)
if len(all_pdf) > 10:
    print(" ...")


In [ ]:
def extract_text_from_pdf(pdf_path):
    """Читает PDF постранично, возвращает список строк по страницам."""
    doc = fitz.open(pdf_path)
    pages = [doc.load_page(i).get_text("text") for i in range(len(doc))]
    doc.close()
    return pages


def clean_lines(page_text):
    lines = []
    for line in page_text.split("\n"):
        line = line.replace('\xa0', ' ')
        line = re.sub(r'\s+', ' ', line).strip()
        if line:
            lines.append(line)
    return lines


def normalize_text(pages):
    return ' '.join(pages).replace('\n', ' ').replace('\xa0', ' ').strip()


def parse_habr_page(page_text):
    """
    Разбирает первую страницу PDF-статьи Хабра.
    Возвращает словарь с основными полями для CSV.
    """
    lines = clean_lines(page_text)

    complexity_words = {'Простой', 'Средний', 'Сложный'}
    article_type_words = {
        'Обзор', 'Кейс', 'Туториал', 'Мнение', 'Дайджест',
        'Перевод', 'Ретроспектива', 'Новость', 'Интервью'
    }
    service_lines = {'РЕ КЛА МА', 'РЕКЛАМА', 'При поддержке', 'Объяснить с', 'Подборка курсов от Хабра'}

    time_re = re.compile(r'^\d+\s*мин$')
    views_re = re.compile(r'^\d+(?:[.,]\d+)?\s*[KkКкMmМм]?$')
    ago_re = re.compile(
        r'^\d+\s+(?:секунд(?:у|ы)?|минут(?:у|ы)?|мин|час(?:а|ов)?|день|дня|дней|недел(?:ю|и|ь)|месяц(?:а|ев)?|год|года|лет)\s+назад$',
        re.IGNORECASE,
    )

    result = {
        'Author': '',
        'PublishTime': '',
        'Title': '',
        'Complexity': '',
        'ReadTime': '',
        'Views': '',
        'Hubs': '',
        'ArticleType': '',
        'NameCompany': '',
        'Rating': '',
        'Activity': '',
    }

    for marker in ('Общий рейтинг', 'Оценка'):
        if marker in lines:
            idx = lines.index(marker)
            if idx > 0:
                result['Rating'] = lines[idx - 1]
            if idx + 1 < len(lines):
                result['NameCompany'] = lines[idx + 1]
            if idx + 2 < len(lines):
                activity = lines[idx + 2]
                if activity.lower() != 'подписаться':
                    result['Activity'] = activity
            break

    time_idx = None
    for i, line in enumerate(lines):
        if ago_re.match(line):
            result['PublishTime'] = line
            time_idx = i
            if i > 0:
                result['Author'] = lines[i - 1]
            break

    if time_idx is None:
        return result

    meta_start = None
    for i in range(time_idx + 1, len(lines)):
        line = lines[i]
        if line in complexity_words or time_re.match(line):
            meta_start = i
            break

    if meta_start is None:
        return result

    title_lines = [line for line in lines[time_idx + 1:meta_start] if line not in service_lines]
    result['Title'] = ' '.join(title_lines).strip()

    i = meta_start
    if i < len(lines) and lines[i] in complexity_words:
        result['Complexity'] = lines[i]
        i += 1

    if i < len(lines) and time_re.match(lines[i]):
        result['ReadTime'] = lines[i]
        i += 1

    if i < len(lines) and views_re.match(lines[i]):
        result['Views'] = lines[i]
        i += 1

    hubs_parts = []
    article_types = []

    while i < len(lines):
        line = lines[i]
        if line in service_lines:
            i += 1
            continue
        if line in article_type_words:
            article_types.append(line)
            i += 1
            continue
        hubs_parts.append(line)
        i += 1

    result['Hubs'] = ', '.join(hubs_parts).strip(', ')
    result['ArticleType'] = ', '.join(article_types)
    return result



In [ ]:
# Основной цикл обработки PDF
records = []
brak = []

for pdf_path in all_pdf:
    filename = os.path.basename(pdf_path)
    print(f"\n{'=' * 60}")
    print(f"Файл: {filename}")

    try:
        pages = extract_text_from_pdf(pdf_path)
        parsed = parse_habr_page(pages[0] if pages else '')
        full_text = normalize_text(pages)

        print(f"  Автор       : {parsed['Author'] or '—'}")
        print(f"  Время       : {parsed['PublishTime'] or '—'}")
        print(f"  Заголовок   : {parsed['Title'] or '—'}")
        print(f"  Сложность   : {parsed['Complexity'] or '—'}")
        print(f"  Чтение      : {parsed['ReadTime'] or '—'}")
        print(f"  Просмотры   : {parsed['Views'] or '—'}")
        print(f"  Хабы        : {parsed['Hubs'] or '—'}")
        print(f"  Тип статьи  : {parsed['ArticleType'] or '—'}")
        print(f"  Компания    : {parsed['NameCompany'] or '—'}")
        print(f"  Рейтинг     : {parsed['Rating'] or '—'}")

        records.append({
            'FileName': filename,
            'Author': parsed['Author'],
            'PublishTime': parsed['PublishTime'],
            'Title': parsed['Title'],
            'Complexity': parsed['Complexity'],
            'ReadTime': parsed['ReadTime'],
            'Views': parsed['Views'],
            'Hubs': parsed['Hubs'],
            'ArticleType': parsed['ArticleType'],
            'NameCompany': parsed['NameCompany'],
            'Rating': parsed['Rating'],
            'Activity': parsed['Activity'],
            'TextArticle': full_text,
        })
    except Exception as e:
        print(f"  ОШИБКА: {e}")
        brak.append(pdf_path)

print(f"\n{'=' * 60}")
print(f"Готово: {len(records)} статей | Ошибок: {len(brak)}")
if brak:
    print('Не обработаны:')
    for item in brak:
        print(' ', item)


In [ ]:
# Сохраняем CSV
column_order = [
    'FileName', 'Author', 'PublishTime', 'Title', 'Complexity',
    'ReadTime', 'Views', 'Hubs', 'ArticleType', 'NameCompany',
    'Rating', 'Activity', 'TextArticle'
]

df = pd.DataFrame(records)
df = df.reindex(columns=column_order)
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"Сохранено -> {OUTPUT_CSV} ({len(df)} строк x {len(df.columns)} столбцов)")
df.head()


In [ ]:
# Проверка — читаем CSV обратно
check_df = pd.read_csv(OUTPUT_CSV)
check_df.head()
